In [ ]:
"""
Agent Monitoring Module

This module provides monitoring capabilities for tracking agent decisions,
reasoning, and tool calls in production.
"""

from typing import Dict, List, Optional, Any
from datetime import datetime


class AgentMonitor:
    """
    Monitor for tracking agent decisions, reasoning, and tool calls.

    This class provides logging capabilities for agent activities.
    """

    def __init__(self):
        """Initialize monitor with empty logs and context"""
        self._logs: List[Dict] = []
        self._context: Dict[str, Any] = {}

    def set_context(self, **kwargs):
        """
        Set context for logging (e.g., session_id, user_id).

        Args:
            **kwargs: Context key-value pairs
        """
        self._context.update(kwargs)

    def log_decision(self, decision_type: str, decision: str, reasoning: str = "",
                     evidence: List[str] = None, confidence: float = None):
        """
        Log an agent decision with reasoning and evidence.

        Args:
            decision_type: Type of decision (e.g., "search", "add_to_cart", "checkout")
            decision: The decision made
            reasoning: Reasoning behind the decision
            evidence: List of evidence supporting the decision
            confidence: Confidence score (0-1)
        """
        if evidence is None:
            evidence = []

        entry = {
            "timestamp": datetime.now().isoformat(),
            "type": "decision",
            "decision_type": decision_type,
            "decision": decision,
            "reasoning": reasoning,
            "evidence": evidence,
            "confidence": confidence,
        }
        self._emit(entry)

    def log_reasoning(self, thought: str, observation: Optional[str] = None):
        """
        Log agent reasoning/thought process.

        Args:
            thought: The thought/reasoning to log
            observation: Optional observation from the environment
        """
        log_entry = {
            "timestamp": datetime.now().isoformat(),
            "type": "reasoning",
            "thought": thought,
            "observation": observation[:500] if observation else None
        }
        self._emit(log_entry)

    def log_tool_call(self, tool: str, arguments: Dict[str, Any], result: Any = None,
                      duration: float = None, success: bool = True):
        """
        Log a tool/UCP API call with complete information.

        Args:
            tool: Name of the tool/API called
            arguments: Arguments passed to the tool
            result: Result returned by the tool
            duration: Time taken for the call in seconds
            success: Whether the call was successful
        """
        entry = {
            "timestamp": datetime.now().isoformat(),
            "type": "tool_call",
            "tool": tool,
            "arguments": arguments,
            "result": result,
            "duration": duration,
            "success": success
        }
        self._emit(entry)

    def _emit(self, entry: Dict):
        """
        Emit a log entry (internal method).

        Args:
            entry: Log entry dictionary
        """
        entry.update(self._context)
        self._logs.append(entry)

    def get_logs(self) -> List[Dict]:
        """
        Get all log entries.

        Returns:
            List of log entry dictionaries
        """
        return self._logs.copy()

    def clear_logs(self):
        """Clear all logs (for testing)"""
        self._logs.clear()
        # Note: Don't clear context - it should persist across clear_logs() calls


In [ ]:
"""
Commerce Metrics Tracking Module

This module provides metrics tracking for the e-commerce shopping agent.
"""

from typing import Dict, List, Optional
from datetime import datetime


class CommerceMetrics:
    """
    Track commerce metrics for shopping agent.

    This class tracks searches, cart operations, checkouts, and conversion rates.
    """

    def __init__(self):
        """Initialize metrics with empty counters"""
        self._searches: List[Dict] = []
        self._cart_operations: List[Dict] = []
        self._checkouts: List[Dict] = []
        self._decisions: List[Dict] = []

    def record_product_search(self, query: str, result_count: int, success: bool = True):
        """
        Record a product search operation.

        Args:
            query: Search query string
            result_count: Number of results returned
            success: Whether the search was successful
        """
        self._searches.append({
            "query": query,
            "result_count": result_count,
            "success": success,
            "timestamp": datetime.now().isoformat()
        })

    def record_cart_operation(self, operation: str, product_id: str, success: bool = True):
        """
        Record a cart operation (add, remove, update).

        Args:
            operation: Type of operation (e.g., "add", "remove", "update")
            product_id: ID of the product
            success: Whether the operation was successful
        """
        self._cart_operations.append({
            "operation": operation,
            "product_id": product_id,
            "success": success,
            "timestamp": datetime.now().isoformat()
        })

    def record_checkout(self, checkout_id: str, success: bool, amount: Optional[float] = None):
        """
        Record a checkout attempt.

        Args:
            checkout_id: ID of the checkout
            success: Whether the checkout was successful
            amount: Optional checkout amount
        """
        self._checkouts.append({
            "checkout_id": checkout_id,
            "success": success,
            "amount": amount,
            "timestamp": datetime.now().isoformat()
        })


    def record_decision(self, decision_type: str, decision: str, success: bool = True):
        """
        Record a shopping decision.

        Args:
            decision_type: Type of decision
            decision: The decision made
            success: Whether the decision was successful
        """
        self._decisions.append({
            "decision_type": decision_type,
            "decision": decision,
            "success": success,
            "timestamp": datetime.now().isoformat()
        })


    def get_conversion_rate(self) -> float:
        """
        Calculate conversion rate (searches that led to successful checkouts).

        Returns:
            Conversion rate as a float (0.0 to 1.0)
        """
        successful_searches = sum(1 for s in self._searches if s["success"])
        successful_checkouts = sum(1 for c in self._checkouts if c["success"])

        if successful_searches == 0:
            return 0.0

        return successful_checkouts / successful_searches

    def get_summary(self) -> Dict:
        """
        Get summary statistics.

        Returns:
            Dictionary with:
            - total_searches
            - total_cart_operations
            - total_checkouts
            - successful_checkouts
            - conversion_rate
        """
        return {
            "total_searches": len(self._searches),
            "total_cart_operations": len(self._cart_operations),
            "total_checkouts": len(self._checkouts),
            "successful_checkouts": sum(1 for c in self._checkouts if c["success"]),
            "conversion_rate": self.get_conversion_rate()
        }

    def clear(self):
        """Clear all metrics (for testing)"""
        self._searches.clear()
        self._cart_operations.clear()
        self._checkouts.clear()
        self._decisions.clear()


In [ ]:
"""
E-Commerce Shopping Agent
"""

from typing import Dict, List, Optional, Any
import json
import time
from datetime import datetime

from monitor import AgentMonitor
from metrics import CommerceMetrics
from llm import get_llm_client


class MockUCPClient:
    """Mock UCP client for testing."""

    def __init__(self):
        self._products = {
            "prod_001": {"name": "Wireless Headphones", "price": 99.99, "category": "electronics"},
            "prod_002": {"name": "Running Shoes", "price": 129.99, "category": "sports"},
            "prod_003": {"name": "Coffee Maker", "price": 79.99, "category": "home"},
            "prod_004": {"name": "Laptop Stand", "price": 49.99, "category": "electronics"},
            "prod_005": {"name": "Yoga Mat", "price": 29.99, "category": "sports"},
        }
        self._cart: Dict[str, int] = {}
        self._should_fail = False

    async def search_products(self, query: str) -> List[Dict]:
        if self._should_fail:
            raise Exception("UCP API error: Service unavailable")
        query_lower = query.lower()
        results = []
        for prod_id, product in self._products.items():
            if query_lower in product["name"].lower() or query_lower in product["category"].lower():
                results.append({"id": prod_id, **product})
        return results

    async def add_to_cart(self, product_id: str, quantity: int = 1) -> Dict:
        if self._should_fail:
            raise Exception("UCP API error: Cart service unavailable")
        if product_id not in self._products:
            raise ValueError(f"Product not found: {product_id}")
        self._cart[product_id] = self._cart.get(product_id, 0) + quantity
        return {"success": True, "product_id": product_id, "quantity": self._cart[product_id]}

    async def checkout(self, payment_method: str) -> Dict:
        if self._should_fail:
            raise Exception("UCP API error: Payment service unavailable")
        if not self._cart:
            raise ValueError("Cart is empty")
        total = sum(self._products[pid]["price"] * qty for pid, qty in self._cart.items())
        checkout_id = f"checkout_{int(time.time())}"
        self._cart.clear()
        return {"success": True, "checkout_id": checkout_id, "amount": total}


class ShoppingAgent:
    """E-Commerce Shopping Agent with UCP integration."""

    def __init__(self, session_id: str, user_id: Optional[str] = None):
        self.session_id = session_id
        self.user_id = user_id
        self.ucp_client = MockUCPClient()
        self.monitor = AgentMonitor()
        self.metrics = CommerceMetrics()
        self.llm = get_llm_client()

        self.monitor.set_context(
            session_id=self.session_id,
            user_id=self.user_id
        )

    async def search_products(self, query: str) -> List[Dict]:
        """Search for products using UCP. Returns a list of product dicts."""
        self.monitor.log_reasoning(
            thought=f"User wants to search for: {query}",
            observation=f"Initiating product search for '{query}'"
        )

        self._log_decision(
            decision_type="search",
            decision=query,
            reasoning="User initiated product search",
        )

        start_time = time.time()

        try:
            results = await self.ucp_client.search_products(query)
        except Exception as exc:
            self._handle_error(exc, "search_products")
            return []

        duration = time.time() - start_time

        self._log_tool_call(
            tool="search_products",
            arguments={"query": query},
            result=results,
            duration=duration,
            success=bool(results)
        )

        self.metrics.record_product_search(
            query=query,
            result_count=len(results),
            success=bool(results)
        )

        self.monitor.log_reasoning(
            thought=f"Search completed for '{query}'",
            observation=f"Found {len(results)} results"
        )

        return results

    async def add_to_cart(self, product_id: str, quantity: int = 1) -> Dict[str, Any]:
        """Add product to cart using UCP."""
        self._log_decision(
            decision_type="add_to_cart",
            decision=product_id,
            reasoning="User added product to cart",
            evidence=[f"Quantity: {quantity}"]
        )

        start_time = time.time()

        try:
            result = await self.ucp_client.add_to_cart(product_id, quantity)
        except Exception as exc:
            return self._handle_error(exc, "add_to_cart")

        duration = time.time() - start_time

        self._log_tool_call(
            tool="add_to_cart",
            arguments={"product_id": product_id, "quantity": quantity},
            result=result,
            duration=duration,
            success=result.get("success", False)
        )

        self.metrics.record_cart_operation(
            operation="add",
            product_id=product_id,
            success=result.get("success", False)
        )

        return result

    async def checkout(self, payment_method: str = "card") -> Dict[str, Any]:
        """Complete checkout using UCP."""
        self._log_decision(
            decision_type="checkout",
            decision=payment_method,
            reasoning="User initiated checkout",
            evidence=[]
        )

        start_time = time.time()

        try:
            result = await self.ucp_client.checkout(payment_method)
        except Exception as exc:
            return self._handle_error(exc, "checkout")

        duration = time.time() - start_time

        self._log_tool_call(
            tool="checkout",
            arguments={"payment_method": payment_method},
            result=result,
            duration=duration,
            success=result.get("success", False)
        )

        self.metrics.record_checkout(
            checkout_id=result.get("checkout_id", "unknown"),
            success=result.get("success", False),
            amount=result.get("amount")
        )

        return result

    def _log_decision(self, decision_type: str, decision: str, reasoning: str,
                      evidence: List[str] = None, confidence: float = None):
        self.monitor.log_decision(
            decision_type=decision_type,
            decision=decision,
            reasoning=reasoning,
            evidence=evidence,
            confidence=confidence
        )

    def _log_tool_call(self, tool: str, arguments: Dict, result: Any,
                       duration: float, success: bool):
        self.monitor.log_tool_call(
            tool=tool,
            arguments=arguments,
            result=result,
            duration=duration,
            success=success
        )

    def _handle_error(self, error: Exception, operation: str) -> Dict[str, Any]:
        self.monitor.log_reasoning(
            thought=f"Error occurred during {operation}: {error}",
            observation=f"Operation: {operation}, Error: {error}"
        )
        self.metrics.record_decision(
            decision_type="error",
            decision=str(error),
            success=False
        )
        return {
            "success": False,
            "error": str(error),
            "operation": operation
        }